In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os
from pathlib import Path

# Load .env from the root folder
BASE_DIR = Path.cwd().parent
load_dotenv(dotenv_path=BASE_DIR / ".env")

# Connect to Neon
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Test connection
with engine.connect() as conn:
    print("✅ Connected to Neon PostgreSQL successfully!")

✅ Connected to Neon PostgreSQL successfully!


In [2]:
def run_query(sql_query):
    """Execute a SQL query and return a Pandas DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(sql_query, conn)

print("✅ Helper function ready.")

✅ Helper function ready.


In [3]:
query_kpis = """
SELECT 
    ROUND(SUM("TotalAmount")::numeric, 2) AS total_revenue,
    COUNT(DISTINCT "Invoice") AS total_orders,
    COUNT(DISTINCT "CustomerID") AS total_customers,
    SUM("Quantity") AS total_units_sold,
    ROUND((SUM("TotalAmount") / COUNT(DISTINCT "Invoice"))::numeric, 2) AS avg_order_value
FROM retail_transactions;
"""

df_kpis = run_query(query_kpis)
print(df_kpis.to_string(index=False))

 total_revenue  total_orders  total_customers  total_units_sold  avg_order_value
   17374804.27         36969             5878        10513952.0           469.98


In [4]:
query_monthly = """
SELECT 
    DATE_TRUNC('month', "InvoiceDate")::date AS month,
    ROUND(SUM("TotalAmount")::numeric, 2) AS monthly_revenue,
    COUNT(DISTINCT "Invoice") AS orders
FROM retail_transactions
GROUP BY DATE_TRUNC('month', "InvoiceDate")
ORDER BY month;
"""

df_monthly = run_query(query_monthly)
df_monthly['month'] = pd.to_datetime(df_monthly['month'])

fig = px.line(
    df_monthly, 
    x='month', 
    y='monthly_revenue', 
    title='📈 Monthly Revenue Trend',
    markers=True
)
fig.update_layout(xaxis_title='Month', yaxis_title='Revenue (£)')
fig.show()

In [5]:
query_top_products = """
SELECT 
    "Description",
    SUM("Quantity") AS units_sold,
    ROUND(SUM("TotalAmount")::numeric, 2) AS revenue
FROM retail_transactions
GROUP BY "Description"
ORDER BY revenue DESC
LIMIT 10;
"""

df_top = run_query(query_top_products)

fig = px.bar(
    df_top, 
    x='revenue', 
    y='Description', 
    orientation='h',
    title='🏆 Top 10 Products by Revenue',
    text='revenue',
    color='revenue',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=500)
fig.show()